### Phase 5 (Evaluation)

# Arviointi

Tässä vaiheessa astutaan taaksepäin ja arvioidaan koko projekti liiketoimintanäkökulmasta. 
Phase 4:ssä arvioitiin mallien teknistä suorituskykyä — nyt kysytään: **ratkaisimmeko 
oikean ongelman oikealla tavalla?** Lisäksi käydään läpi koko prosessi vaihe vaiheelta 
ja päätetään jatkosta.


## Tulosten arviointi liiketoimintatavoitteita vasten

Projektin alkuperäinen tavoite oli rakentaa malli, joka ennustaa **käytetyn auton 
myyntihinnan** aineiston ominaisuuksien perusteella. Alla arvioidaan, täyttyivätkö 
tämä tavoite.

---

### Hintamalli (price) — CatBoost (optimoidut parametrit: depth=9, l2_leaf_reg=2)

| Kriteeri | Tavoite | Saavutettu | Arvio |
|----------|:-------:|:----------:|:-----:|
| R² ≥ 0.88 | 0.88 | **0.9614** | ✅ Ylitetty |
| MAE ≤ $3 000 | $3 000 | **$2 136** | ✅ Ylitetty |
| ≥ 85 % ennusteista ±$5 000 tarkkuudella | 85 % | **90.8 %** | ✅ Ylitetty |
| MAE alle $10 000 autoilla ≤ $2 000 | $2 000 | $1 186 | ⚠️ Harhaanjohtava |
| MAE yli $75 000 autoilla ≤ $10 000 | $10 000 | $5 935 | ✅ Ylitetty |

**Johtopäätös:** Hintamalli täyttää asetetut kriteerit. Halpisautojen segmentti ($0–$10 000) täyttää kriteerin luvuilla, mutta tulos on **harhaanjohtava käytännössä**: `brand_model_avg_price` vetää ennustetta 
ylöspäin vanhoille yksilöille, koska aggregaatti lasketaan kaikista ikäluokista. 

Liiketoiminnan näkökulmasta malli on **käyttökelpoinen hinnoittelun tukityökaluna** 
kunhan tiedostaa mallin rajoitteet ja virhemarginaalin. Ostaja voi tarkistaa onko hinta markkinahinnan mukainen, ja myyjä voi käyttää mallia hinnoittelupäätösten tukena.

## Daysonmarket-mallin hylkääminen — perustelut

Projektin alkuvaiheessa suunniteltiin kahta mallia: hintamallin lisäksi 
**myynnissäoloaikamallia (daysonmarket)**. Arviointiprosessin aikana tehtiin 
päätös jättää daysonmarket-arviointi kokonaan pois. Perustelut ovat seuraavat:

---

### 1. Aineiston väärä tulkinta dataan tutustumisvaiheessa

`daysonmarket`-muuttuja ei kuvaa myyntiaikaa kuten aluksi virheellisesti pääteltiin. Se kertoo ainoastaan kuinka monta päivää auto on ollut myynnissä sillä hetkellä kun aineisto on kerätty crawlerilla **datankeruuhetkellä 14.9. - 17.9.2020**. Aineiston avulla ei pysty vastaamaan auton myyntiaikaan koska sitä tietoa ei ole aineistossa eikä sitä siitä pysty päättelemään. Aineistossa olevat autot ovat siis kaikki myymättömiä, mikä on jälkikäteen ajateltuna ilmiselvää kun aineisto on kerätty myynti-ilmoituksista, ei siellä juurikaan tapaa olla myytyjen autojen tietoja. 


### 2. 

> **Päätös:** Daysonmarket-malli jätetään pois. Projekti keskittyy hintamalliin, 
> jonka liiketoiminnallinen arvo on selkeä ja aineistopohja luotettava.


## Prosessin arviointi

Käydään läpi projektin vaiheet ja arvioidaan mikä toimi hyvin ja mitä voisi 
tehdä toisin.

---

### Phase 1–2: Datan ymmärtäminen ja esikäsittely

**Mikä toimi:**
- Raakadata (2,6M riviä) käsiteltiin tehokkaasti Polars-kirjastolla
- Puuttuvat arvot imputoitiin asianmukaisesti segmenttikohtaisesti
- Datavuodon riski tunnistettiin ajoissa (`avg_dom`-featuret)

**Mitä voisi tehdä toisin:**
- 'daysonmarket' kentän rooli olisi pitänyt tajuta heti
- ajallinen painottuminen noin 13 kk ajanjaksoon (noin 98%) olisi pitänyt huomioida selkeämmin ja mahdollisesti rajata käsittely ainoastaan ko. ajanjaksoon

---

### Phase 3: Feature Engineering

**Mikä toimi:**
- Markkina-tason aggregaatit (`brand_model_avg_price`) olivat mallinnuksen 
  tärkein yksittäinen feature — juuri niin kuin intuitiivisesti odotettiin
- Varusteflagit (navigaatio, nahkapenkit, kattoluukku) tunnistettiin oikein 
  ja näkyivät feature importanceissa
- Feature engineering oli kokonaisuutena onnistunut: mallin tärkeimmät 
  featuret ovat sellaisia joita etukäteen osattiin odottaa, mikä viittaa 
  siihen että datan logiikka ymmärrettiin oikein

**Mitä voisi tehdä toisin:**
- **`brand_model_avg_price` lasketaan kaikista ikäluokista** — aggregaatti
  sisältää uudet ja vanhat yksilöt sekaisin, mikä vetää ennustetta ylöspäin
  vanhoille ja paljon ajetuille autoille. Esimerkiksi Ford Focus -aggregaatti
  sisältää 2020-mallit, jolloin 2005-vuosimallin hinta yliarvioidaan
  systemaattisesti. Ikä- tai kilometriluokittainen aggregaatti palvelisi
  halpisautojen segmenttiä paremmin.
- **Varusteflagit absoluuttisina eikä suhteellisina** — 22 binääriflagia
  kertoo onko varuste olemassa, mutta ei onko auto hyvin varusteltu
  *suhteessa muihin saman merkin ja mallin autoihin*. Merkki/malli-kohtainen
  normaalistus (`options_count / avg_options_by_model`) olisi voinut kuvata
  varusteiden hinnoitteluvaikutusta realistisemmin.
- **Ääripäiden segmentit vaativat erilaisia menetelmiä** — sekä halpisautojen
  ($0–$10k) että premium-autojen ($75k+) hinnoittelu nojaa tekijöihin joita
  yleinen feature-setti ei tavoita riittävästi. Segmenttikohtainen feature
  engineering tai erilliset mallit olisivat tarpeellisia näitä varten.

---

### Phase 4: Mallinnus ja optimointi

**Mikä toimi:**
- Kolmen mallin (XGBoost, CatBoost, LightGBM) vertailu antoi luotettavan 
  kuvan suorituskyvystä
- CatBoost valittiin parhaaksi malliksi 
- Residuaalianalyysi paljasti konkreettisesti missä malli epäonnistuu 
  (premium-segmentti)
- Hyväksymiskriteerit asetettiin etukäteen, mikä teki arvioinnista objektiivista
- Manuaalinen parametriviritys (`depth=9`, `l2_leaf_reg=2`) paransi MAE:ta 
  yli prosentin

**Mitä voisi tehdä toisin:**
- K-fold-ristiinvalidointi tehtiin jälkikäteen vain CatBoostille — vastaava
  analyysi XGBoostille ja LightGBM:lle puuttuu, joten mallien välinen
  ero (~$80 MAE) voi selittyä osittain satunnaisvaihtelulla
- Halppis- sekä premium-segmenteille olisi voinut rakentaa erilliset mallit

---

### Yhteenveto prosessin laadusta

Prosessi noudatti CRISP-DM-metodologiaa johdonmukaisesti. Merkittävin 
prosessipäätös — daysonmarket-mallin hylkääminen aineiston rakenteellisten 
ongelmien takia — oli perusteltu ja osoittaa kykyä tunnistaa milloin malli 
ei sovi tarkoitukseensa. Tekninen toteutus oli huolellinen ja dokumentoitu.

## Mallien analysointi

### Mallien vertailu — hintamalli

| Malli | MAE ($) | RMSE ($) | R² | MAPE (%) | Huomio |
|-------|--------:|---------:|---:|---------:|--------|
| XGBoost | 2 217 | 3 326 | 0.959 | 8.43 | Suosi `mileage_bin`-ryhmittelyä |
| **CatBoost** | **2 136** | **3 216** | **0.961** | **8.23** | **Paras kaikilla mittareilla** |
| LightGBM | 2 328 | 3 432 | 0.956 | 9.05 | Korkein RMSE — herkempi poikkeamille |

**CatBoost valitaan tuotantomallksi** seuraavilla perusteilla:
- Paras suorituskyky kaikilla neljällä mittarilla
- Sisäänrakennettu kategoristen muuttujien käsittely — vähentää 
  esikäsittelytarvetta
- Robustius ylisovittumista vastaan suurella aineistolla
- Parametrioptimointiin hyvin reagoiva: `depth=9` ja `l2_leaf_reg=2` 
  paransivat sekä MAE:ta että RMSE:tä 

---

### Miksi CatBoost suoriutui parhaiten? — spekulaatio

Ero mallien välillä on pieni, joten ei ole yhtä selkeää syytä — mutta 
muutama tekijä selittää todennäköisesti CatBoosin etua:

**Kategoriset muuttujat.** Aineistossa on useita tärkeitä kategorisia 
featureja kuten merkki, malli ja osavaltio. CatBoost käsittelee nämä 
natiivisti ordered target encoding -menetelmällä, kun taas XGBoost ja 
LightGBM vaativat manuaalisen enkoodauksen (tai jättävät hyödyntämättä 
kategorisen rakenteen kokonaan). Koska `brand_model_avg_price` on 
ylivoimaisesti tärkein feature ja se on kategorinen aggregaatti, 
CatBoosin kyky käsitellä kategorisia muuttujia tarkemmin saattaa 
selittää eron.

**Ordered boosting.** CatBoost käyttää ordered boosting -tekniikkaa, 
joka vähentää target leakagea harjoitusdatassa. Suurella aineistolla 
(2,6M riviä) tämä voi antaa pienen mutta johdonmukaisen edun — 
harjoitusdata ei pääse "vuotamaan" mallin sovittamiseen samalla tavalla.

**Symmetriset puut.** CatBoost rakentaa oletuksena symmetrisiä 
päätöspuita, joissa kaikki haarat samalla tasolla käyttävät samaa 
split-kriteeriä. Tämä on laskennallisesti tehokkaampi ja saattaa 
yleistää paremmin tasaisesti jakautuneessa aineistossa kuin 
epäsymmetriset puut (XGBoost, LightGBM).

> Tämä on veikkaustasolla — mallien välinen ero (~$190 MAE) on niin pieni
> että satunnaisvaihtelu tai hyperparametrien valinta voisi selittää sen
> yhtä hyvin. K-fold ristiinvalidointi tehtiin CatBoostille ja vahvisti
> sen tuloksen luotettavuuden, mutta vastaava analyysi XGBoostille ja
> LightGBM:lle puuttuu.

---

### Segmenttikohtainen suorituskyky

| Hintaluokka | MAE ($) | Käytännön soveltuvuus |
|-------------|--------:|----------------------|
| $0–$10 000 | $1 186 | ⚠️ Harhaanjohtava — katso huomio |
| $10 000–$20 000 | $1 343 | ✅ Erittäin hyvä |
| $20 000–$30 000 | $1 699 | ✅ Hyvä |
| $30 000–$50 000 | $2 502 | ✅ Hyvä — päämarkkina |
| $50 000–$75 000 | $4 833 | ⚠️ Kohtalainen |
| $75 000+ | $5 935 | ⚠️ Kohtalainen |

Malli toimii parhaiten siellä missä dataa on eniten — alle $50 000 autoilla, 
jotka kattavat valtaosan käytettyjen autojen markkinasta.

**Huomio $0–$10 000 segmentistä:** MAE $1 336 näyttää hyvältä, mutta segmentti 
on käytännössä epäluotettava. Syy on `brand_model_avg_price`-featuressa, joka 
dominoi mallin ennustetta: se laskee merkin ja mallin keskihinnan koko aineistosta, 
jossa painottuvat uudemmat ja kalliimmat yksilöt. Esimerkiksi Ford Focus -aggregaatti 
vedetään ylöspäin uudempien mallien takia, jolloin malli yliarvioi systemaattisesti 
vanhan, paljon ajetun yksilön hinnan (esim. vm. 2005, 200 000 mailia). Halpisautojen 
hinnoitteluun tarvittaisiin erillinen malli, joka on koulutettu pelkästään tämän 
hintaluokan autoilla.

## Hyperparametrien optimointi — kokemukset

Mallin parametreja optimoitiin kahdella lähestymistavalla. 
Tulokset ja yksityiskohtainen koodi löytyvät erillisestä 
dokumentista `Optimointi_Testaus_Phase4.ipynb`.

---

### Mitä tehtiin

**Optuna-haku (Bayesian optimization, 50 trialia, ~5h):**
Automaattinen haku 500k rivin otoksella 5-fold CV:llä. Paras löydetty 
konfiguraatio (depth=10, lr=0.142, iterations=1400) osoittautui 
koko testijoukolle huonommaksi kuin baseline: MAE kasvoi $124 
ja RMSE $190.

**Manuaalitestit Optuna-oppien perusteella:**
Kolme kohdennettua testiä baseline-konfiguraatiosta. V2 (`depth=9`, 
`l2_leaf_reg=2`) osoittautui parhaaksi: MAE -$34 ja RMSE -$61 
suhteessa baselineen.

---

### Oliko optimoinnista hyötyä?

Kyllä — mutta vain manuaalinen testaus tuotti tuloksen. 
Automaattinen Optuna-optimointi epäonnistui:

| Lähestymistapa | MAE muutos | Tulos |
|----------------|:----------:|-------|
| Optuna (500k otos) | +$124 | ❌ Heikensi |
| Manuaali V2 | -$34 | ✅ Paransi |

**Parannus on aito mutta maltillinen** — 1.4% MAE:ssa ja 1.2% RMSE:ssä. 
Se vastaa noin 34 dollarin pienempää tyypillistä ennustevirhettä, mikä 
on käytännön hinnoittelun kannalta marginaalinen.

---

### Mitä optimointi opetti

**Optuna epäonnistui rakenteellisesta syystä:** 500k rivin otos ei edusta 
2,6M rivin datan parametriavaruutta riittävästi. Aggressiivinen konfiguraatio 
joka toimii pienemmällä otoksella ylisovittuu koko dataan. 
Lisäksi `bagging_temperature` ei vaikuttanut lainkaan 
(vaatisi `bootstrap_type='Bayesian'` eksplisiittisesti), joten Optuna 
optimoi parametria jolla ei ollut vaikutusta.

**Manuaaliset testit olivat tehokkaampia** juuri sen takia, että ne 
perustuivat ymmärrykseen: Optuna halusi syvempiä puita ja vähemmän 
regularisaatiota — tämä tieto sovellettuna konservatiivisesti 
(depth 8→9, l2 3→2) koko aineistoon tuotti oikean suunnan ilman 
ylisovittumista.

**Pieni parannus kovalla työllä** — noin 5 tuntia Optuna-hakua + 
manuaalitestit tuottivat alle 1.5% parannuksen. Sama työmäärä 
feature engineeringissä tai lisädatassa tuottaisi todennäköisesti 
suuremman parannuksen. Hyperparametrivirityksen arvo on rajallinen 
kun baseline on jo hyvin konfiguroitu.


## Varusteflagien epälooginen hinnoittelu — havainto ja analyysi

Varusteflagien vaikutusta malliin tutkittiin manuaalisesti syötämällä sama auto
eri varustelutasoilla ja seuraamalla miten ennuste muuttuu. Testi paljasti
selkeän epäloogisuuden:

### Havaittu ilmiö

| Varustelutaso | Ennuste |
|---|---:|
| Ei varusteita | $33 500 |
| + Navigaatio | $34 300 |
| + Navigaatio + Sunroof | $32 600 |
| Kaikki varusteet | $32 000 |

Navigaatio nosti ennustetta odotetusti (+$800), mutta sunroofin lisääminen
*laski* ennustetta $1 700 — eli varusteet eivät käyttäydy additiivisesti.
Kaikkien varusteiden lisääminen tuotti **alhaisimman** ennusteen, vaikka
oikeassa markkinassa täysin varusteltu auto on lähes aina arvokkaampi kuin
perusversio.

Osassa autoja varusteet toimivat oikeaan suuntaan (pienehkö positiivinen
vaikutus), mutta epäloogisuus toistui useammassa tapauksessa.

---

### Mahdolliset syyt

**1. Varusteet absoluuttisina — ei suhteellisina**

Malli ei tiedä onko varuste tavallinen vai poikkeuksellinen kyseiselle
merkille ja mallille. Sunroof voi olla standardi Volvo V90:ssä mutta harvinaisuus
vanhassa Toyota Corollassa. Kun sunroof esiintyy Corollan yhteydessä, malli
ei osaa tulkita sitä premium-signaalina — se vain vastaa sille opittua
keskiarvoa, jossa sunroof saattaa korreloida halvempien autojen kanssa
(esim. vanha eurooppalainen hatchback).

**2. Varusteyhdistelmät ohjaavat mallin eri haaroihin**

CatBoost rakentaa päätöspuita, joissa useat featuret yhdessä ohjaavat
ennustetta. Kun navigaatio + sunroof esiintyvät yhtä aikaa, malli saattaa
reitittää ennusteen haaraan, jossa tämä yhdistelmä on opittu halvempien
autojen kanssa (esim. tietyn aikakauden mid-range autot joissa oli molemmat).
Sama pätee kaikkien varusteiden yhdistelmälle.

**3. Navigaatio vs. sunroof — eri asema markkinassa**

Navigaatio korreloi moderneihin autoihin (CarPlay, Android Auto yhdistyvät
usein samaan pakettiin) ja nostaa hintaa loogisesti. Sunroof sen sijaan on
yleinen myös 2000-luvun alun mid-range autoissa — jolloin se voi korreloida
*negatiivisesti* hinnan kanssa, koska sunroof-autot ovat keskimäärin vanhempia
tai halvemmista segmenteistä kuin navigaatioautot.

**4. `brand_model_avg_price` dominoi — varusteet marginaalisessa roolissa**

Koska `brand_model_avg_price` selittää ylivoimaisesti suurimman osan
ennusteesta, yksittäisten varusteiden vaikutus on pieni ja altis
epälineaarisille vuorovaikutuksille. Pieni muutos useammassa varusteessa
voi siirtää ennustetta eri puuhaaran alueelle, jossa opittu "tyypillinen
hinta" on eri tasolla kuin intuitio edellyttäisi.

**5. Harvinainen varusteluyhdistelmä — vähän opetusdataa**

Jos tietyllä merkki+malli-yhdistelmällä on harvoja havaintoja kaikkien
varusteiden kombinaatiolle, malli yleistää virheellisesti muiden autojen
datasta. Tulos on satunnainen eikä seuraa markkinalogiikkaa.

---

### Yhteys ablation-testeihin

Ablation-testit (`Optimointi_Testaus_Phase4.ipynb`) osoittivat, että
varusteiden poistaminen kokonaan heikensi MAE:ta noin **+$300**.
Varusteet siis *auttavat* mallia globaalisti — mutta yksittäisen auton
tasolla ne voivat käyttäytyä epäloogisesti yllä kuvatusta syistä.

Tämä on klassinen jännitys ML-malleissa: **aggregaattitasolla feature
parantaa suorituskykyä, mutta yksittäisten havaintojen tasolla sen
vaikutus voi olla satunnainen tai käänteinen**.

---

### Johtopäätös

Varusteflagit toimivat parhaiten silloin kun varustelukombinkaatio on
tyypillinen saman merkin ja mallin muille autoille. Epätavalliset
yhdistelmät tuottavat epäluotettavia tuloksia. Tämä tukee Phase 3:n
"voisi tehdä toisin" -havaintoa: absoluuttisten flagien sijaan
**suhteellinen varustenormaalistus** (`options_count / avg_options_by_model`)
kuvaisi varustetason hinnoitteluvaikutusta realistisemmin.


## K-fold ristiinvalidointi — tulokset

5-fold CV ajettiin koko aineistolla (2,6M riviä) V2-parametreilla varmistaaksemme,
että Phase 4:n yksittäinen train/test-jako antoi luotettavan kuvan mallin suorituskyvystä.
Yksityiskohtainen koodi ja tulokset löytyvät `Optimointi_Testaus_Phase4.ipynb`-tiedostosta.

| | MAE | RMSE | R² | <$5k |
|--|----:|-----:|---:|-----:|
| K-fold CV (keskiarvo) | $1 888 | $2 706 | 0.9664 | 93.7% |
| K-fold Std | $3 | $10 | 0.0003 | 0.1% |
| Phase 4 test-joukko | $2 136 | $3 216 | 0.961 | 90.8% |

**Std MAE $3** osoittaa että malli on erittäin vakaa — tulos ei riipu siitä miten
data sattuu jakautumaan. Phase 4:n $2 461 ei ole sattumaa.

CV-tulos ($1 888) on selvästi parempi kuin Phase 4:n tulos, mutta **ylioptimistinen**:
aggregaattifeaturet (`brand_model_avg_price` ym.) on laskettu koko aineistosta etukäteen,
jolloin ne sisältävät jo validointifoldin tietoja — klassinen tietovuoto-ongelma
ristiinvalidoinnissa. CV-tulosta ei käytetä virallisena suorituskykymittarina.

> **Johtopäätös:** K-fold vahvisti Phase 4:n tuloksen luotettavuuden.
> Mallissa ei ole havaittavaa ylisovittumista.

## Seuraavat askeleet — päätös jatkosta

### Suositus: Etenee deployment-vaiheeseen rajoituksin

Hintamalli täyttää asetetut hyväksymiskriteerit normaalin hintaluokan autoille 
ja on valmis deployment-vaiheeseen seuraavilla rajauksilla:

- **Soveltamisalue:** $10 000–$75 000 käytetyt autot
- **Käyttötarkoitus:** hinnoittelun tukityökalu, ei automaattinen hinnoittelija
- **Dokumentoitava rajoite:** alle $10 000 ja yli $75 000 segmentit eivät kuulu 
  luotettavan ennusteen piiriin

---

### Prioriteettijärjestyksessä — jatkokehitys

| Prioriteetti | Kehityskohde | Perustelu |
|:---:|---|---|
| 🟡 Keskitaso | Varustetasojen yksinkertaistus deploymentissa | Korvaa 22 yksittäistä varusteflagia kolmella tasolla (low / medium / high) — helpottaa pikaista hinta-arviota ilman kaikkien varusteiden erittelyä |
| 🟡 Keskitaso | Merkki/malli-kohtainen varustenormaalistus | Sen sijaan että kysytään "onko sunroof", lasketaan kuinka hyvin varusteltu auto on verrattuna saman merkin ja mallin muihin autoihin (`options_relative_score = options_count / avg_options_by_model`) |
| 🟢 Matala | Erillinen halpisautojen malli ($0–$10 000) | Nykyinen malli on epäluotettava tässä segmentissä — `brand_model_avg_price` vetää ennustetta ylöspäin, koska aggregaatti lasketaan kaikista ikäluokista |
| 🟢 Matala | Erillinen premium-malli ($75 000+) | Aineistoa suhteellisen vähän ja nykyinen malli huomio ilmeisen huonosti tämän segmentin hintatekijät |

---

### Loppuarvio

Projekti saavutti tavoitteensa: **käytetyn auton hintaa voidaan ennustaa 
koneoppimisen keinoin luotettavasti** tavallisessa hintaluokassa. Prosessi 
noudatti hyvää metodologiaa — datavuoto tarkistettiin, rajoitteet tunnistettiin 
rehellisesti, daysonmarket-mallin hylkääminen oli perusteltu päätös 
aineiston rakenteellisten ongelmien takia, ja parametrioptimointia kokeiltiin 
rehellisesti tulokset dokumentoiden.

Malli on valmis deployment-vaiheeseen määritellyillä rajauksilla.